# AI Tutor (MCP + OpenAI Agents SDK)

An AI tutor built on the **Model Context Protocol (MCP)** and the **OpenAI Agents SDK**, where the agent reaches tools/resources through an MCP server.

**What it demonstrates**
- Connecting an agent to an MCP server (SSE) for tools and context
- Building agents with the OpenAI Agents SDK
- Wrapping it in a Gradio interface

**Stack:** Python · MCP · OpenAI Agents SDK · Gradio


In [55]:
import os
import requests
import httpx
import gradio as gr
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown, Image
from openai import OpenAI
from PIL import Image
import  asyncio, pathlib

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)

MODEL_NAME = "gpt-4o-mini"

In [56]:
def print_markdown(text):
    display(Markdown(text))

In [7]:
!pip install openai-agents

   ---------------------------------------- 0.0/856.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/856.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/856.1 kB ? eta -:--:--
   ------------------------ --------------- 524.3/856.1 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 856.1/856.1 kB 2.0 MB/s  0:00:00

   ---------------------------------------- 0/2 [griffelib]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   -------------------- ------------------- 1/2 [openai-agents]
   ---------------------------------------- 2/2 [openai

In [58]:
MCP_BASE = "http://localhost:7860/gradio_api/mcp/sse"


from agents.mcp import MCPServerSse

mcp_tool = MCPServerSse(
    params={
        "url": MCP_BASE,   # 'url', not 'base_url'
        "timeout": 30,
    },
    name="AI Tutor",                        # separate kwarg
    client_session_timeout_seconds=60,      # separate kwarg
)


In [59]:
EXPLANATION_LEVELS = {
    1: "Like I'm a 5 year old",
    2: "Like I'm a 10 year old",
    3: "Like a high school student",
    4: "Like a college student",
    5: "Like an expert in the field",
}

In [61]:
from typing import Generator

def explain_concept(question: str, level: int) -> Generator[str, None, None]:
    """Stream an explanation of *question* at the requested *level* (1‑5). If 1, explanation would be like we are talking to a 5 year old and if 5, explanation would be technical and complex."""
    if not question.strip():
        yield "Error: Question cannot be blank."
        return

    level_description = EXPLANATION_LEVELS.get(level, "Clearly and Concisely")
    system_prompt = "You are an AI tutor. Explain the following concept " f"{level_description}."
    _stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        stream=True,
        temperature=0.8,
    )
    partial = ""
    for chunk in _stream:
        delta = getattr(chunk.choices[0].delta, "content", None)
        if delta:
            partial += delta
            yield partial

In [63]:
def summarise_text(text:str, compression_rate: float = 0.3) -> Generator[str, None, None]:
    """Stream a summary of *text* at the requested *compression_rate* leanth"""

    if not text.strip():
        yield "Error: Text cannot be blank."
        return
    
    ratio = max(0.1, min(compression_rate, 0.8))
    system_prompt = (
        "You are a world class summariser. Reduce the following text to about"
        f" {int(ratio*100)}% of its original length, while preserving key information."
    )
    _stream = client.chat.completions.create(
        model=MODEL_NAME,   
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ],
        stream=True,
        temperature=0.5,
    )
    partial = ""
    for chunk in _stream:
        delta = getattr(chunk.choices[0].delta, "content", None)
        if delta:
            partial += delta
            yield partial

In [64]:
def generate_flashcards(topic: str, num_cards: int = 10) -> Generator[str, None, None]:
    """Stream *num_cards* Q/A flashcards for *topic* in JSON lines format."""

    if num_cards < 1 or num_cards > 20:
        yield "Error: Number of flashcards must be between 1 and 20."
        return
    if not topic.strip():
        yield "Error: Topic cannot be blank."
        return
    
    system_prompt = (
        "You are an AI that generates study flashcards"
        'Return each flashcard on its own line as a JSON: {"q": <question>, "a": <answer>}.'
    )
    user_prompt = f"Create {num_cards} flashcards about {topic}."

    _stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
        temperature=0.8,
    )
    partial = ""
    for chunk in _stream:
        delta = getattr(chunk.choices[0].delta, "content", None)
        if delta:
            partial += delta
            yield partial

In [65]:
def quiz_me(topic: str, level: int = 3, num_questions: int = 5) -> Generator[str, None, None]:
    """Stream a quiz with numbered Qs then reveal answers after all questions."""
    if num_questions < 1 or num_questions > 20:
        yield "Error: Number of questions must be between 1 and 20."
        return
    if not topic.strip():
        yield "Error: Topic cannot be blank."
        return

    level_description = EXPLANATION_LEVELS.get(level, "at an intermediate level")

    system_prompt = (
        "You are an AI quiz master. Generate a quiz of multiple choice questions "
        f"about {topic} {level_description}. Number the questions. After listing all Qs"
        "add an \nANSWER KEY section with the correct answers."
    )

    _stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
        ],
        stream=True,
        temperature=0.7,
    )
    partial = ""
    for chunk in _stream:
        delta = getattr(chunk.choices[0].delta, "content", None)
        if delta:
            partial += delta
            yield partial

In [66]:
def build_demo():
    with gr.Blocks() as demo:
        gr.Markdown("# AI Tutor MCP Toolkit – Demo Console")
        with gr.Tab("Explain Concept"):
            q = gr.Textbox(label="Concept / Question")
            lvl = gr.Slider(1, 5, value=3, step=1, label="Explanation Level")
            out1 = gr.Markdown()
            gr.Button("Explain").click(explain_concept, inputs=[q, lvl], outputs=out1)
        with gr.Tab("Summarize Text"):
            txt = gr.Textbox(lines=8, label="Long Text")
            ratio = gr.Slider(0.1, 0.8, value=0.3, step=0.05, label="Compression Ratio")
            out2 = gr.Markdown()
            gr.Button("Summarize").click(summarise_text, inputs=[txt, ratio], outputs=out2)
        with gr.Tab("Flashcards"):
            topic_fc = gr.Textbox(label="Topic")
            n_fc = gr.Slider(1, 20, value=5, step=1, label="# Cards")
            out3 = gr.Markdown()
            gr.Button("Generate").click(generate_flashcards, inputs=[topic_fc, n_fc], outputs=out3)
        with gr.Tab("Quiz Me"):
            topic_q = gr.Textbox(label="Topic")
            lvl_q = gr.Slider(1, 5, value=3, step=1, label="Difficulty Level")
            n_q = gr.Slider(1, 15, value=5, step=1, label="# Questions")
            out4 = gr.Markdown()
            gr.Button("Start Quiz").click(quiz_me, inputs=[topic_q, lvl_q, n_q], outputs=out4)
    return demo


if __name__ == "__main__":
    print("Starting AI Tutor MCP Toolkit on port 7860…")
    build_demo().launch(server_name = "0.0.0.0", mcp_server = True)
# --- END OF FILE ---

Starting AI Tutor MCP Toolkit on port 7860…
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.

🔨 Launching MCP server:
* Streamable HTTP URL: http://localhost:7860/gradio_api/mcp/


In [54]:
gr.close_all()

Closing server running on port: 7860


In [67]:
client = httpx.Client()

def fetch_schema(server_url):
    """Fetches and parses the MCP schema from a server."""

    print(f"Fetching MCP schema from {server_url}...")

    response = client.get(server_url, timeout=60)
    response.raise_for_status()
    schema_data = response.json()
    print("Schema fetched successfully.")

    return schema_data

In [69]:
MCP_BASE2 = "http://localhost:7860/gradio_api/mcp/schema"

print("Fetching AI Tutor MCP schema...")

tutor_schema = fetch_schema(MCP_BASE2)

if tutor_schema:
    print("\nAI Tutor Schema Content:")

    print(json.dumps(tutor_schema, indent=2))

print("\n" + "=" * 50 + "\n")

Fetching AI Tutor MCP schema...
Fetching MCP schema from http://localhost:7860/gradio_api/mcp/schema...
Schema fetched successfully.

AI Tutor Schema Content:
[
  {
    "name": "explain_concept",
    "description": "Stream an explanation of *question* at the requested *level* (1\u20115). If 1, explanation would be like we are talking to a 5 year old and if 5, explanation would be technical and complex.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": ""
        },
        "level": {
          "type": "number",
          "description": "",
          "default": 3
        }
      }
    },
    "meta": {
      "file_data_present": false,
      "mcp_type": "tool",
      "endpoint_name": "explain_concept"
    }
  },
  {
    "name": "summarise_text",
    "description": "Stream a summary of *text* at the requested *compression_rate* leanth",
    "inputSchema": {
      "type": "object",
      "properti